## GPT prompting: n80 10k examples, second test

### requires python >= 3.10

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [3]:
from prompts.semantic_categories.v02.prompt import SYSTEM_PROMPT, FEW_SHOTS_STR, FEW_SHOTS

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
RESULTS_DIR = "../../results/"

EXAMPLE_FILE = "n80_examples_large_v01/gpt_v01/" + "gpt_10K_b12_run01.csv"

GPT_ANSWER_FILE = "n80_examples_large_v01/gpt_v02/"+ "gpt_10K_b10_run01.csv"

CONF_FILE = 'azure.ini'

# OSA I : Andmed


## testimise põhjusel on kasutusel vana 10k v1 andmefail, et tulemusi saaks võrrelda

In [4]:
df = pd.read_csv(RESULTS_DIR+EXAMPLE_FILE, encoding="utf-8",  sep=",")

In [5]:
len(df)

10000

In [6]:
# kui faili on laused salvestatud shufflitud olekus, siis võiks võtta lihtsalt esimesed n

spatial_obl_ex = df.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

In [7]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
4569,11982198,jaamas,jaam,põrkama,kokku,in,7464499,Hiinlased põlesid küünlana Teist ja viimast k...,NaN,NaN,NaN,yes,NaN
9579,4630979,eetris,eeter,jälgima,NaN,in,2888224,Tipphetkel jälgis 7. oktoobril eetris olnud sa...,NaN,NaN,NaN,no,The word 'eetris' refers to broadcasting or be...
5482,18243478,Zumaias,Zumaia,õhkima,NaN,in,11385201,Kaksteist tundi hiljem õhkisid oletatavad ETA ...,NaN,NaN,NaN,yes,NaN
2258,11996016,juhatuses,juhatus,levitama,NaN,in,7473322,Liitmisjuttu levitavad peamiselt Tammemäe suur...,NaN,NaN,NaN,no,The word 'juhatuses' indicates a governing bod...
6755,12694462,Margareetasse,Margareeta,mahtuma,ära,ill,7923778,"Mina ütlesin : "" Paksu Margareetasse ei mahu n...",NaN,NaN,PER,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,7099192,doktoriõppesse,doktoriõpe,toimuma,NaN,ill,4414313,Arstiteaduse doktoriõppesse toimuvad augustiku...,NaN,NaN,NaN,no,The phrase 'doktoriõppesse' refers to a progra...
5235,8284638,pingevaeselt,pingevaene,kulgema,NaN,abl,5163938,"Viimased poolfinaalid kulgesid pingevaeselt , ...",NaN,NaN,NaN,no,The word 'pingevaeselt' describes a state or q...
4630,13436270,autosse,auto,investeerima,NaN,ill,8384613,Uude autosse investeerib Opel ligikaudu 300 mi...,NaN,NaN,NaN,yes,NaN
8168,3095400,Haapsalus,Haapsalu,kukkuma,alla,in,1942040,Haapsalus aga kukkus oma koduaias tiiki alla k...,NaN,location,LOC,yes,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [7]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [14]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [4]:
SYSTEM_PROMPT

'\nYou are a classification assistant.\nIn this task location refers to "adverbial of place" (Estonian: kohamäärus) or "locative adverb".\nYour task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" functions as a location in the context of the sentence.\nAdverbial of place answers to the question “where” (kus?/kuhu?/kust?) in the context of the sentence.\nIt is a place or concept where something or someone is located, goes to or comes from.\nCriteria:\n- concrete place (bank, table, Berlin)\n- abstract (literature, soul, TV channels, government, top of a group, history, thought, domain)\n- inanimate (journal, chair, wifi, bag, medal, toy, food, computer, wire, body parts)\n- alive (mother, Peter, dog, doctor, teacher)\n- event (dress rehearsal, camp, class, situation, meeting)\n- state or condition conceptualized as space (life, trouble, consciousness, attitude)\n- Locations ARE NOT phrases that show time, state of being, owner, experience

In [6]:
#FEW_SHOTS_STR

## Andmete söötmine

In [ ]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":   json.dumps(user_payload, ensure_ascii=False) }
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [ ]:

def explain_non_locations(
    client, 
    deployment,
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0,

) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" +  json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]
    #return None, None,None 
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices


In [17]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [18]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'."
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'."
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'."
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'."
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN


In [ ]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, FEW_SHOTS_STR, SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio,
            client=client, 
            deployment=DEPLOYMENT
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= 4500000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    
    #break



In [32]:
#used_tokens # batch 10-> 3700,  10K lauset -> ~ 7.9 eur

3726

In [21]:
#len(results)

10000

## andmed tabelisse 

### enne kontroll kas andmeid on puudu ja vastavad lüngad täita

In [22]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification2"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation2"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification2"] = new_results
    df["explanation2"] = new_explanations

    #df["explanation"] = new_explanations 

In [23]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'.",yes,
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN,yes,
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'.",yes,
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'.",yes,
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'.",yes,
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN,yes,


### salvestada tulemused faili

In [24]:
saving_fname = RESULTS_DIR+GPT_ANSWER_FILE 

In [25]:
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## optional saving

fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)

In [9]:
#df = pd.read_csv(RESULTS_DIR+GPT_ANSWER_FILE , encoding="utf-8",  sep=",")

In [13]:
print("yes:", len(df[df["classification2"]=="yes"])) 
print("yes explained:", len(df[(df["classification2"]=="yes") & (~df["explanation2"].isna())]))
print("no:", len(df[df["classification2"]=="no"])) 

yes: 8694
yes explained: 1251
no: 1306
